In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

# Data Dictionary: `diabetic_data.csv`

**Dataset:** Diabetes 130-US hospitals for years 1999–2008 (UCI Machine Learning Repository)
**Size:** 101,766 rows (hospital encounters) × 50 columns, covering 71,518 unique patients
**Target variable:** `readmitted`


## 1. Identifiers

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `encounter_id` | Integer | Unique ID of a hospital admission (encounter) | One per row (101,766 unique) |
| `patient_nbr` | Integer | Unique ID of a patient | 71,518 unique; patients can appear in several rows (repeat admissions), so deduplicate before train/test splitting to avoid leakage |

## 2. Demographics

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `race` | Categorical | Patient's race | `Caucasian`, `AfricanAmerican`, `Hispanic`, `Asian`, `Other`, `?` (2,273 missing) |
| `gender` | Categorical | Patient's gender | `Female`, `Male`, `Unknown/Invalid` (only 3 rows) |
| `age` | Categorical (ordinal) | Age group in 10-year bins | `[0-10)`, `[10-20)`, … `[90-100)` (10 groups) |
| `weight` | Categorical (ordinal) | Weight in pounds, binned | `[0-25)`, `[25-50)`, … `>200`, `?`. **About 97% missing** (98,569 rows), so it is usually dropped |

## 3. Admission & Discharge Information

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `admission_type_id` | Integer (coded) | How the patient was admitted | 8 codes, e.g. 1 = Emergency, 2 = Urgent, 3 = Elective, 4 = Newborn, 5/6/8 = Not available / NULL, 7 = Trauma Center. See the `IDs_mapping.csv` file from UCI |
| `discharge_disposition_id` | Integer (coded) | Where the patient went after discharge | 26 codes, e.g. 1 = Discharged to home, 3 = Skilled nursing facility, 6 = Home with home health service, 11/13/14/19/20/21 = Expired or hospice, 18 = NULL. Codes for death/hospice should usually be removed when modeling readmission |
| `admission_source_id` | Integer (coded) | Where the patient was admitted from | 17 codes, e.g. 1 = Physician referral, 7 = Emergency room, 4 = Transfer from hospital |
| `time_in_hospital` | Integer | Length of stay in days | Range 1–14 |
| `payer_code` | Categorical | Insurance / payer of the encounter | 17 payer types (e.g. `MC` Medicare, `MD` Medicaid, `HM` HMO, `SP` Self-pay) plus `?` (40,256 missing) |
| `medical_specialty` | Categorical | Specialty of the admitting physician | 72 specialties (e.g. `InternalMedicine`, `Cardiology`, `Emergency/Trauma`) plus `?` (49,949 missing) |

## 4. Utilization & Procedures

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `num_lab_procedures` | Integer | Number of lab tests performed during the encounter | Count |
| `num_procedures` | Integer | Number of procedures (other than lab tests) performed during the encounter | Range 0–6 |
| `num_medications` | Integer | Number of distinct generic medications administered during the encounter | Count |
| `number_outpatient` | Integer | Number of outpatient visits of the patient in the **year before** the encounter | Count |
| `number_emergency` | Integer | Number of emergency visits of the patient in the **year before** the encounter | Count |
| `number_inpatient` | Integer | Number of inpatient visits of the patient in the **year before** the encounter | Count |

## 5. Diagnoses

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `diag_1` | Categorical | Primary diagnosis, coded as the first three digits of ICD-9 | 717 distinct codes; codes starting with `250` are diabetes mellitus; `V` and `E` codes are supplementary classifications |
| `diag_2` | Categorical | Secondary diagnosis (ICD-9) | 749 distinct codes, `?` if none |
| `diag_3` | Categorical | Additional secondary diagnosis (ICD-9) | 790 distinct codes, `?` if none |
| `number_diagnoses` | Integer | Number of diagnoses entered in the system for the encounter | Range 1–16 |

## 6. Lab Results

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `max_glu_serum` | Categorical | Glucose serum test result | `>200`, `>300`, `Norm`, `None` (not tested; 96,420 rows) |
| `A1Cresult` | Categorical | HbA1c (glycated hemoglobin) test result, a measure of average blood sugar over about 3 months | `>8` (greater than 8%), `>7` (greater than 7%, less than 8%), `Norm` (less than 7%), `None` (not tested; 84,748 rows) |

## 7. Diabetes Medications (24 columns)

Each column below records whether that drug was prescribed during the encounter and whether the dosage changed. All share the same coding:

| Value | Meaning |
|---|---|
| `No` | The drug was not prescribed |
| `Steady` | The drug was prescribed and the dosage did not change |
| `Up` | The dosage was increased during the encounter |
| `Down` | The dosage was decreased during the encounter |

| Column | Type | Drug / Class | Notes |
|---|---|---|---|
| `metformin` | Categorical | Metformin (biguanide) | Most common first-line drug |
| `repaglinide` | Categorical | Repaglinide (meglitinide) | |
| `nateglinide` | Categorical | Nateglinide (meglitinide) | |
| `chlorpropamide` | Categorical | Chlorpropamide (sulfonylurea) | Very rare |
| `glimepiride` | Categorical | Glimepiride (sulfonylurea) | |
| `acetohexamide` | Categorical | Acetohexamide (sulfonylurea) | Almost never used (`No` / `Steady` only) |
| `glipizide` | Categorical | Glipizide (sulfonylurea) | |
| `glyburide` | Categorical | Glyburide (sulfonylurea) | |
| `tolbutamide` | Categorical | Tolbutamide (sulfonylurea) | Almost never used (`No` / `Steady` only) |
| `pioglitazone` | Categorical | Pioglitazone (thiazolidinedione) | |
| `rosiglitazone` | Categorical | Rosiglitazone (thiazolidinedione) | |
| `acarbose` | Categorical | Acarbose (alpha-glucosidase inhibitor) | |
| `miglitol` | Categorical | Miglitol (alpha-glucosidase inhibitor) | |
| `troglitazone` | Categorical | Troglitazone (thiazolidinedione) | Almost never used (`No` / `Steady` only) |
| `tolazamide` | Categorical | Tolazamide (sulfonylurea) | Almost never used |
| `examide` | Categorical | Examide | **Constant column** (always `No`); safe to drop |
| `citoglipton` | Categorical | Citoglipton (sitagliptin) | **Constant column** (always `No`); safe to drop |
| `insulin` | Categorical | Insulin | One of the most frequently used and informative medication columns |
| `glyburide-metformin` | Categorical | Combination: glyburide + metformin | |
| `glipizide-metformin` | Categorical | Combination: glipizide + metformin | Almost never used |
| `glimepiride-pioglitazone` | Categorical | Combination: glimepiride + pioglitazone | Almost never used |
| `metformin-rosiglitazone` | Categorical | Combination: metformin + rosiglitazone | Almost never used |
| `metformin-pioglitazone` | Categorical | Combination: metformin + pioglitazone | Almost never used |

## 8. Medication Summary Flags

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `change` | Categorical (binary) | Whether there was **any** change in diabetic medications (dosage or drug) during the encounter | `Ch` = change (47,011 rows), `No` = no change (54,755 rows) |
| `diabetesMed` | Categorical (binary) | Whether **any** diabetic medication was prescribed | `Yes` (78,363 rows), `No` (23,403 rows) |

## 9. Target Variable

| Column | Type | Description | Values / Notes |
|---|---|---|---|
| `readmitted` | Categorical | Days until the patient was readmitted to hospital | `<30` = readmitted within 30 days (11,357), `>30` = readmitted after more than 30 days (35,545), `NO` = never readmitted (54,864). Commonly binarized as `<30` vs everything else |

---


# Data Exploration -- > Explore and find mistakes in Data.

In [2]:
df = pd.read_csv("diabetic_data.csv.zip")
df

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.describe().round(2)

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00
mean,1.652016e+08,5.433040e+07,2.02,3.72,5.75,4.40,43.10,1.34,16.02,0.37,0.20,0.64,7.42
std,1.026403e+08,3.869636e+07,1.45,5.28,4.06,2.99,19.67,1.71,8.13,1.27,0.93,1.26,1.93
min,1.252200e+04,1.350000e+02,1.00,1.00,1.00,1.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00
25%,8.496119e+07,2.341322e+07,1.00,1.00,1.00,2.00,31.00,0.00,10.00,0.00,0.00,0.00,6.00
50%,1.523890e+08,4.550514e+07,1.00,1.00,7.00,4.00,44.00,1.00,15.00,0.00,0.00,0.00,8.00
75%,2.302709e+08,8.754595e+07,3.00,4.00,7.00,6.00,57.00,2.00,20.00,0.00,0.00,1.00,9.00
max,4.438672e+08,1.895026e+08,8.00,28.00,25.00,14.00,132.00,6.00,81.00,42.00,76.00,21.00,16.00


In [6]:
df.describe(include= ['object'])

,race,gender,age,weight,payer_code,medical_specialty,diag_1,diag_2,diag_3,max_glu_serum,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
count,101766,101766,101766,101766,101766,101766,101766,101766,101766,5346,...,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766
unique,6,3,10,10,18,73,717,749,790,3,...,1,4,4,2,2,2,2,2,2,3
top,Caucasian,Female,[70-80),?,?,?,428,276,250,Norm,...,No,No,No,No,No,No,No,No,Yes,NO
freq,76099,54708,26068,98569,40256,49949,6862,6752,11555,2597,...,101766,47383,101060,101753,101765,101764,101765,54755,78363,54864


- what does '?' represent 

In [7]:
df.isna().mean().round(4) *100

encounter_id                 0.00
patient_nbr                  0.00
race                         0.00
gender                       0.00
age                          0.00
weight                       0.00
admission_type_id            0.00
discharge_disposition_id     0.00
admission_source_id          0.00
time_in_hospital             0.00
payer_code                   0.00
medical_specialty            0.00
num_lab_procedures           0.00
num_procedures               0.00
num_medications              0.00
number_outpatient            0.00
number_emergency             0.00
number_inpatient             0.00
diag_1                       0.00
diag_2                       0.00
diag_3                       0.00
number_diagnoses             0.00
max_glu_serum               94.75
A1Cresult                   83.28
metformin                    0.00
repaglinide                  0.00
nateglinide                  0.00
chlorpropamide               0.00
glimepiride                  0.00
acetohexamide 

# In depth check for categorical columns

In [8]:
cat_cols = df.select_dtypes(include= 'object')
cat_cols
for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

race
6
['Caucasian' 'AfricanAmerican' '?' 'Other' 'Asian' 'Hispanic']
----------------------------------------------------------------------------------------------------
gender
3
['Female' 'Male' 'Unknown/Invalid']
----------------------------------------------------------------------------------------------------
age
10
['[0-10)' '[10-20)' '[20-30)' '[30-40)' '[40-50)' '[50-60)' '[60-70)'
 '[70-80)' '[80-90)' '[90-100)']
----------------------------------------------------------------------------------------------------
weight
10
['?' '[75-100)' '[50-75)' '[0-25)' '[100-125)' '[25-50)' '[125-150)'
 '[175-200)' '[150-175)' '>200']
----------------------------------------------------------------------------------------------------
payer_code
18
['?' 'MC' 'MD' 'HM' 'UN' 'BC' 'SP' 'CP' 'SI' 'DM' 'CM' 'CH' 'PO' 'WC' 'OT'
 'OG' 'MP' 'FR']
----------------------------------------------------------------------------------------------------
medical_specialty
73
['Pediatrics-Endocrinology' '?'

- In depth check for Numerical columns

In [9]:
cat_cols = df.select_dtypes(include= 'number')
cat_cols
for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)
    df.describe().round(2)

encounter_id
101766
[  2278392    149190     64410 ... 443854148 443857166 443867222]
----------------------------------------------------------------------------------------------------
patient_nbr
71518
[  8222157  55629189  86047875 ... 140199494 120975314 175429310]
----------------------------------------------------------------------------------------------------
admission_type_id
8
[6 1 2 3 4 5 8 7]
----------------------------------------------------------------------------------------------------
discharge_disposition_id
26
[25  1  3  6  2  5 11  7 10  4 14 18  8 13 12 16 17 22 23  9 20 15 24 28
 19 27]
----------------------------------------------------------------------------------------------------
admission_source_id
17
[ 1  7  2  4  5  6 20  3 17  8  9 14 10 22 11 25 13]
----------------------------------------------------------------------------------------------------
time_in_hospital
14
[ 1  3  2  4  5 13 12  9  7 10  6 11  8 14]
--------------------------------------

# Data Cleaning

- replace '?' with Nan

In [10]:
df = df.replace('?', np.nan)
df.isna().mean().round(4) *100

encounter_id                 0.00
patient_nbr                  0.00
race                         2.23
gender                       0.00
age                          0.00
weight                      96.86
admission_type_id            0.00
discharge_disposition_id     0.00
admission_source_id          0.00
time_in_hospital             0.00
payer_code                  39.56
medical_specialty           49.08
num_lab_procedures           0.00
num_procedures               0.00
num_medications              0.00
number_outpatient            0.00
number_emergency             0.00
number_inpatient             0.00
diag_1                       0.02
diag_2                       0.35
diag_3                       1.40
number_diagnoses             0.00
max_glu_serum               94.75
A1Cresult                   83.28
metformin                    0.00
repaglinide                  0.00
nateglinide                  0.00
chlorpropamide               0.00
glimepiride                  0.00
acetohexamide 

- for column race replace nan with other for easy tracking

In [11]:
df['race'] = df['race'].replace(np.nan, 'Other')
print(df['race'].unique())

['Caucasian' 'AfricanAmerican' 'Other' 'Asian' 'Hispanic']


- remove unnec columns  
- remove columns has high nan values

In [12]:
df = df.drop(['weight','patient_nbr','encounter_id'],axis=1)



- there's no need for diag_2 or diag_3 columns as main diagnosis is already clarified in diag_1 column

In [13]:
df = df.drop(['diag_2','diag_3'],axis=1)

- remove the missing rows with low frequency 

In [14]:
df.dropna(subset=['race','diag_1'], inplace=True, ignore_index=True)

In [15]:
df.isna().mean().round(4) *100

race                         0.00
gender                       0.00
age                          0.00
admission_type_id            0.00
discharge_disposition_id     0.00
admission_source_id          0.00
time_in_hospital             0.00
payer_code                  39.55
medical_specialty           49.08
num_lab_procedures           0.00
num_procedures               0.00
num_medications              0.00
number_outpatient            0.00
number_emergency             0.00
number_inpatient             0.00
diag_1                       0.00
number_diagnoses             0.00
max_glu_serum               94.75
A1Cresult                   83.28
metformin                    0.00
repaglinide                  0.00
nateglinide                  0.00
chlorpropamide               0.00
glimepiride                  0.00
acetohexamide                0.00
glipizide                    0.00
glyburide                    0.00
tolbutamide                  0.00
pioglitazone                 0.00
rosiglitazone 

 - In both columns nan doesn't mean the value is unknown. It means the test was not performed, which is real information.

In [16]:
df[["max_glu_serum", "A1Cresult"]] = df[["max_glu_serum", "A1Cresult"]].fillna("test not performed")

In [17]:
df.isna().mean().round(4) *100

race                         0.00
gender                       0.00
age                          0.00
admission_type_id            0.00
discharge_disposition_id     0.00
admission_source_id          0.00
time_in_hospital             0.00
payer_code                  39.55
medical_specialty           49.08
num_lab_procedures           0.00
num_procedures               0.00
num_medications              0.00
number_outpatient            0.00
number_emergency             0.00
number_inpatient             0.00
diag_1                       0.00
number_diagnoses             0.00
max_glu_serum                0.00
A1Cresult                    0.00
metformin                    0.00
repaglinide                  0.00
nateglinide                  0.00
chlorpropamide               0.00
glimepiride                  0.00
acetohexamide                0.00
glipizide                    0.00
glyburide                    0.00
tolbutamide                  0.00
pioglitazone                 0.00
rosiglitazone 

In [18]:
# Final cleaned dataset
df.to_csv("cleaned_diabetic_data.csv", index=False)


In [19]:
df = pd.read_csv("cleaned_diabetic_data.csv")

C:\Users\dell\AppData\Local\Temp\ipykernel_25696\2186376694.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_diabetic_data.csv")


# Feature Engineering

In [20]:
df.head(5)

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [21]:
#  Summarize the 23 drug columns into two counts
drug_cols = df.loc[:, "metformin":"metformin-pioglitazone"].columns
meds = df[drug_cols]
df["number_diabetic_drugs"]   = (meds != "No").sum(axis=1)             # how many diabetes drugs
df["number_doses_changed"] = meds.isin(["Up", "Down"]).sum(axis=1)  # how many doses changed

In [22]:
drug_cols

Index(['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone'],
      dtype='object')

In [23]:
df.head(2)

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,number_diabetic_drugs,number_doses_changed
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,NO,0,0
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,No,No,No,No,No,Ch,Yes,>30,1,1


In [24]:
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap to a fixed width

In [25]:
# 3. Group hundreds of ICD-9 diagnosis codes into a few categories
def icd9_group(code):
    if pd.isna(code) or str(code).startswith(("V", "E")): return "Other"
    c = float(code)
    if 250 <= c < 251:               return "Diabetes"
    if 390 <= c <= 459 or c == 785:  return "Circulatory"
    if 460 <= c <= 519 or c == 786:  return "Respiratory"
    if 520 <= c <= 579 or c == 787:  return "Digestive"
    if 580 <= c <= 629 or c == 788:  return "Genitourinary"
    if 800 <= c <= 999:              return "Injury"
    return "Other"

df["Diagnosis_Group"] = df["diag_1"].map(icd9_group)

In [26]:
# make a new cleaned dataframe for analysis 
df_cleaned = df.copy()
df_cleaned = df_cleaned.drop(drug_cols, axis= 1 )
df_cleaned.drop(['admission_source_id','discharge_disposition_id','medical_specialty','diag_1'],axis=1, inplace=True)



In [27]:
df_cleaned.to_csv("cleaned_diabetic_data.csv", index=False)

In [28]:
df_cleaned = pd.read_csv("cleaned_diabetic_data.csv")
df_cleaned

,race,gender,age,admission_type_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,max_glu_serum,A1Cresult,change,diabetesMed,readmitted,number_diabetic_drugs,number_doses_changed,Diagnosis_Group
0,Caucasian,Female,[0-10),6,1,NaN,41,0,1,0,0,0,1,test not performed,test not performed,No,No,NO,0,0,Diabetes
1,Caucasian,Female,[10-20),1,3,NaN,59,0,18,0,0,0,9,test not performed,test not performed,Ch,Yes,>30,1,1,Other
2,AfricanAmerican,Female,[20-30),1,2,NaN,11,5,13,2,0,1,6,test not performed,test not performed,No,Yes,NO,1,0,Other
3,Caucasian,Male,[30-40),1,2,NaN,44,1,16,0,0,0,7,test not performed,test not performed,Ch,Yes,NO,1,1,Other
4,Caucasian,Male,[40-50),1,1,NaN,51,0,8,0,0,0,5,test not performed,test not performed,Ch,Yes,NO,2,0,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101740,AfricanAmerican,Male,[70-80),1,3,MC,51,0,16,0,0,0,9,test not performed,>8,Ch,Yes,>30,2,1,Diabetes
101741,AfricanAmerican,Female,[80-90),1,5,MC,33,3,18,0,0,1,9,test not performed,test not performed,No,Yes,NO,1,0,Digestive
101742,Caucasian,Male,[70-80),1,1,MC,53,0,9,1,0,0,13,test not performed,test not performed,Ch,Yes,NO,2,1,Other
101743,Caucasian,Female,[80-90),2,10,MC,45,2,21,0,0,1,9,test not performed,test not performed,Ch,Yes,NO,3,1,Injury


In [29]:
df_cleaned.isna().mean().round(4)*100

race                      0.00
gender                    0.00
age                       0.00
admission_type_id         0.00
time_in_hospital          0.00
payer_code               39.55
num_lab_procedures        0.00
num_procedures            0.00
num_medications           0.00
number_outpatient         0.00
number_emergency          0.00
number_inpatient          0.00
number_diagnoses          0.00
max_glu_serum             0.00
A1Cresult                 0.00
change                    0.00
diabetesMed               0.00
readmitted                0.00
number_diabetic_drugs     0.00
number_doses_changed      0.00
Diagnosis_Group           0.00
dtype: float64

# Data Analysis

- Readmission distribution

In [30]:
readmit_df = df_cleaned["readmitted"].value_counts().reset_index()
readmit_df.columns = ["OutcomeCode", "Encounters"]
readmit_df["OutcomeLabel"] = readmit_df["OutcomeCode"].map({
                "NO": "No Readmission",
                ">30": "Readmitted > 30 Days",
                "<30": "Early Readmitted (< 30 Days)"
            })
print(readmit_df)
fig_donut = px.pie(
                readmit_df,
                values="Encounters",
                names="OutcomeLabel",
                hole=0.5,
                color="OutcomeLabel",
                color_discrete_map={
                    "No Readmission": "#1F6FB2",
                    "Readmitted > 30 Days": "#60A5FA",
                    "Early Readmitted (< 30 Days)": "#DC2626"
                }
            )
fig_donut

  OutcomeCode  Encounters                  OutcomeLabel
0          NO       54852                No Readmission
1         >30       35541          Readmitted > 30 Days
2         <30       11352  Early Readmitted (< 30 Days)


- Primary Diagnosis Breakdown admissions

In [31]:
diag_counts = df_cleaned["Diagnosis_Group"].value_counts().reset_index()
diag_counts.columns = ["Diagnosis", "Encounters"]
diag_counts = diag_counts.sort_values(by="Encounters", ascending=True)
print(diag_counts)
fig_diag = px.bar(
                diag_counts,
                x="Encounters",
                y="Diagnosis",
                orientation="h",
                text_auto=True,
            )
fig_diag

       Diagnosis  Encounters
6  Genitourinary        5117
5         Injury        6974
4       Diabetes        8757
3      Digestive        9475
2    Respiratory       14423
1          Other       26562
0    Circulatory       30437


- Hospital Length of Stay (Days)

In [32]:
los_counts = df_cleaned["time_in_hospital"].value_counts().sort_values(ascending=False).sort_index().reset_index()
los_counts.columns = ["Days in Hospital", "Encounters"]
print(los_counts)
fig_los = px.bar(
                los_counts,
                x="Days in Hospital",
                y="Encounters",
                text_auto=True,
                     
            )   
fig_los 

    Days in Hospital  Encounters
0                  1       14206
1                  2       17223
2                  3       17750
3                  4       13917
4                  5        9965
5                  6        7537
6                  7        5858
7                  8        4390
8                  9        3002
9                 10        2342
10                11        1855
11                12        1448
12                13        1210
13                14        1042


In [33]:
df


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,number_diabetic_drugs,number_doses_changed,Diagnosis_Group
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,1,test not performed,test not performed,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,0,0,Diabetes
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,9,test not performed,test not performed,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,1,1,Other
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,6,test not performed,test not performed,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,1,0,Other
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,7,test not performed,test not performed,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,1,1,Other
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,5,test not performed,test not performed,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,2,0,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101740,AfricanAmerican,Male,[70-80),1,3,7,3,MC,NaN,51,0,16,0,0,0,250.13,9,test not performed,>8,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30,2,1,Diabetes
101741,AfricanAmerican,Female,[80-90),1,4,5,5,MC,NaN,33,3,18,0,0,1,560,9,test not performed,test not performed,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO,1,0,Digestive
101742,Caucasian,Male,[70-80),1,1,7,1,MC,NaN,53,0,9,1,0,0,38,13,test not performed,test not performed,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,NO,2,1,Other
101743,Caucasian,Female,[80-90),2,3,7,10,MC,Surgery-General,45,2,21,0,0,1,996,9,test not performed,test not performed,No,No,No,No,No,No,Steady,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,3,1,Injury


    # admission type id can be converted to 
1 - emergency 
2 - urgent 
3- elective 
4- newborn 
7- trauma center 
568-  other

In [34]:
#  convert admission type to 6 categories 1 - emergency 2 - urgent 3- elective 4- newborn 7- trauma center 568-  other

df_cleaned['admission_type_id'] = df_cleaned['admission_type_id'].replace({1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4: 'Newborn', 7: 'Trauma Center', 5: 'Other', 6: 'Other', 8: 'Other'})
# make a bar chart to show the distribution of admission types
admission_type = df_cleaned["admission_type_id"].value_counts().reset_index()
admission_type.columns = ["admission_type_id", "Encounters"]
admission_type = admission_type.sort_values(by="Encounters", ascending=True)
print(admission_type)
fig_adm = px.bar(
                admission_type,
                x="admission_type_id",
                y="Encounters",
                text_auto=True,
            )
fig_adm

  admission_type_id  Encounters
5           Newborn          10
4     Trauma Center          21
3             Other       10395
2            Urgent       18478
1          Elective       18862
0         Emergency       53979


- Patient Admissions by Age Bracket  

In [35]:
#Patient Admissions by Age Bracket
age_summary = df_cleaned["age"].value_counts().sort_values(ascending=False).sort_index().reset_index()  
age_summary.columns = ["Age Bracket", "Encounters"]
print(age_summary)
fig_age = px.bar(
                age_summary,
                x="Age Bracket",
                y="Encounters",
                text_auto=True,
            )
fig_age

  Age Bracket  Encounters
0      [0-10)         161
1     [10-20)         691
2     [20-30)        1656
3     [30-40)        3775
4     [40-50)        9683
5     [50-60)       17254
6     [60-70)       22476
7     [70-80)       26064
8     [80-90)       17192
9    [90-100)        2793


- Gender Distribution & Readmission Comparison

In [36]:
#Gender Distribution & Readmission Comparison 
gender_df = df_cleaned.groupby(["gender", "readmitted"]).size().reset_index(name="Encounters")
gender_df.columns = ["Gender", "Readmission Status", "Encounters"]
#remove the "Unknown/Invalid" gender category
gender_df = gender_df[gender_df["Gender"] != "Unknown/Invalid"]
print(gender_df)
fig_gender = px.bar(
                gender_df,
                x="Gender",
                y="Encounters",
                color="Readmission Status",
                barmode="group",
                text_auto=True,
            )
fig_gender

   Gender Readmission Status  Encounters
0  Female                <30        6151
1  Female                >30       19517
2  Female                 NO       29035
3    Male                <30        5201
4    Male                >30       16024
5    Male                 NO       25814


- Hospital Admissions by Racial Cohort

In [37]:
# make a bar chart to show the distribution of racial cohorts
race_df = df_cleaned["race"].value_counts().reset_index()
race_df.columns = ["Race", "Encounters"]
print(race_df)
fig_race = px.bar(
                race_df,
                x="Race",
                y="Encounters",
                color="Race",
                color_discrete_map={
                    "Caucasian": "#1F6FB2",
                    "AfricanAmerican": "#60A5FA",
                    "Asian": "#DC2626",
                    "Hispanic": "#FACC15",
                    "Other": "#10B981"
                }
            )
fig_race

              Race  Encounters
0        Caucasian       76092
1  AfricanAmerican       19201
2            Other        3777
3         Hispanic        2035
4            Asian         640


- HbA1c Blood Sugar Test Ordering

In [38]:
a1c_counts = df["A1Cresult"].value_counts().reset_index()
a1c_counts.columns = ["ResultCode", "Encounters"]
a1c_counts["ResultLabel"] = a1c_counts["ResultCode"].map({
                "test not performed": "Untested (No Lab)",
                ">8": "Elevated (>8%)",
                "Norm": "Normal",
                ">7": "Moderately High (>7%)"
            })
print(a1c_counts)
fig_a1c_donut = px.pie(
                a1c_counts,
                values="Encounters",
                names="ResultLabel",
                hole=0.55,
                color="ResultLabel",
                color_discrete_map={
                    "Untested (No Lab)": "#93C5FD",
                    "Normal": "#10B981",
                    "Moderately High (>7%)": "#F59E0B",
                    "Elevated (>8%)": "#DC2626"
                }
            )
fig_a1c_donut

           ResultCode  Encounters            ResultLabel
0  test not performed       84730      Untested (No Lab)
1                  >8        8214         Elevated (>8%)
2                Norm        4989                 Normal
3                  >7        3812  Moderately High (>7%)


- HbA1c Testing vs 30-Day Readmission Rate

In [48]:
#HbA1c Testing vs 30-Day Readmission Rate chart
a1c_readmit_df = df_cleaned.groupby(["A1Cresult", "readmitted"]).size().reset_index(name="Encounters")
a1c_readmit_df.columns = ["A1Cresult", "Readmission Status", "Encounters"]
print(a1c_readmit_df)
fig_a1c_readmit = px.bar(
                a1c_readmit_df,
                x="A1Cresult",
                y="Encounters",
                barmode="group",
                color="Readmission Status",
                text_auto=True,
            )
fig_a1c_readmit


             A1Cresult Readmission Status  Encounters
0                   >7                <30         383
1                   >7                >30        1300
2                   >7                 NO        2129
3                   >8                <30         811
4                   >8                >30        2901
5                   >8                 NO        4502
6                 Norm                <30         481
7                 Norm                >30        1599
8                 Norm                 NO        2909
9   test not performed                <30        9677
10  test not performed                >30       29741
11  test not performed                 NO       45312
